# MUSIN-G Dataset Loader Demo

This notebook demonstrates how to use the BCMIMusingLoader to load the MUSIN-G dataset,
which contains EEG recordings from 20 subjects listening to 12 different songs.

## Dataset Overview
- **Subjects**: 20
- **Songs**: 12 (various genres)
- **Sessions**: 12 per subject (one per song)
- **Sampling Rate**: 250 Hz
- **Format**: BIDS-compliant EEGLAB .set files
- **Music IDs**: Integer from 0 to 11 (MusingMusicId)

In [7]:
from pathlib import Path
import sys

# Add parent directory to path if needed
project_root = Path.cwd()
# if str(project_root) not in sys.path:
#     sys.path.insert(0, str(project_root))

from eeg_music.bcmi import BCMIMusingLoader, create_bcmi_loader
from eeg_music.data import MusingMusicId, MusicFilename, copy_from_dataloader_into_dir
from eeg_music.data import EEGMusicDataset

## 1. Load MUSIN-G Dataset with BCMIMusingLoader

We can use either the specific loader class or the factory function.

In [8]:
# Path to the musin-g dataset
musin_g_path = Path(".") / "datasets" / "musin_g_data"

# Method 1: Direct instantiation
loader = BCMIMusingLoader(str(musin_g_path))

# Method 2: Using factory function (automatically detects dataset type)
# loader = create_bcmi_loader(str(musin_g_path))

print(f"Dataset: {loader.dataset_name}")
print(f"Number of subjects: {len(loader.subjects)}")
print(f"Subjects: {loader.subjects[:5]}...")  # Show first 5

Dataset: musin-g
Number of subjects: 20
Subjects: ['001', '002', '003', '004', '005']...


In [9]:
loader_export = BCMIMusingLoader(str(musin_g_path))
loader_export.load_all_subjects()
new_dataset_save_dir = Path("./datasets/musin_g_export2")
copy_from_dataloader_into_dir(loader_export, new_dataset_save_dir)

ds = EEGMusicDataset.load_ondisk(new_dataset_save_dir)

Loading subject 001 (musin-g):
  ✗ Song 01 (run 01): File does not exist:
datasets/musin_g_data/sub-001...
  ✗ Song 02 (run 02): File does not exist:
datasets/musin_g_data/sub-001...
  ✗ Song 03 (run 03): File does not exist:
datasets/musin_g_data/sub-001...
  ✗ Song 04 (run 04): File does not exist:
datasets/musin_g_data/sub-001...
  ✗ Song 05 (run 05): File does not exist:
datasets/musin_g_data/sub-001...
  ✗ Song 06 (run 06): File does not exist:
datasets/musin_g_data/sub-001...
  ✗ Song 07 (run 07): File does not exist:
datasets/musin_g_data/sub-001...
  ✗ Song 08 (run 08): File does not exist:
datasets/musin_g_data/sub-001...
  ✗ Song 09 (run 09): File does not exist:
datasets/musin_g_data/sub-001...
  ✗ Song 10 (run 10): File does not exist:
datasets/musin_g_data/sub-001...
  ✗ Song 11 (run 11): File does not exist:
datasets/musin_g_data/sub-001...
  ✗ Song 12 (run 12): File does not exist:
datasets/musin_g_data/sub-001...
Loading subject 002 (musin-g):
  ✗ Song 01 (run 01): File

In [4]:
ds

NameError: name 'ds' is not defined

## 7. Load Exported Dataset with EegMusicDataset

After exporting, we can load the dataset using EegMusicDataset.

In [ ]:
# Assuming you've exported the dataset in step 6
# exported_path = output_dir

# Load with EegMusicDataset
# dataset = EegMusicDataset(base_dir=exported_path)
# print(f"Loaded {len(dataset)} trials")

# Access a trial
# trial = dataset[0]
# print(f"\nFirst trial:")
# print(trial.pretty())

## 8. Access Behavioral Data

The MUSIN-G dataset includes behavioral ratings (enjoyment and familiarity) in the stimuli directory.

In [ ]:
import pandas as pd

# Load behavioral data
behavioral_file = musin_g_path / "stimuli" / "Behavioural_data"
behavioral_df = pd.read_csv(behavioral_file, sep="\t")

print("Behavioral data columns:", behavioral_df.columns.tolist())
print(f"\nShape: {behavioral_df.shape}")
print("\nFirst few rows:")
print(behavioral_df.head(10))

# Get ratings for subject 1, song 1
subject_1_song_1 = behavioral_df[(behavioral_df['Subject'] == 1) & (behavioral_df['Song_ID'] == 1)]
if not subject_1_song_1.empty:
    enjoyment = subject_1_song_1['Enjoyment'].iloc[0]
    familiarity = subject_1_song_1['Familiarity'].iloc[0]
    print(f"\nSubject 1, Song 1:")
    print(f"  Enjoyment: {enjoyment}/5")
    print(f"  Familiarity: {familiarity}/5")

## 9. Song Information

The musing.py module contains detailed information about all 12 songs.

In [ ]:
from eeg_music.musing import songs_info_enhanced

print("\nSong Information:")
print("=" * 80)
for song_id, info in songs_info_enhanced.items():
    print(f"\nSong {song_id}: {info['name']}")
    print(f"  Artist: {info['artist']}")
    print(f"  Genre: {info['genre']}")
    print(f"  Duration: {info['duration']}s")
    print(f"  Tempo: {info['tempo']} BPM" if info['tempo'] else "  Tempo: N/A")
    print(f"  Characteristics: {info['characteristics']}")

## Summary

This notebook demonstrated:

1. **Loading MUSIN-G dataset** with `BCMIMusingLoader`
2. **Understanding `MusingMusicId`** type (integers 1-12 for 12 songs)
3. **Accessing EEG data** for subjects and sessions
4. **Iterating over trials** with `trial_iterator()`
5. **Exporting dataset** with `copy_from_dataloader_into_dir()`
6. **Loading exported data** with `EegMusicDataset`
7. **Accessing behavioral ratings** (enjoyment and familiarity)
8. **Song metadata** from the musing module

### Key Features of MUSIN-G Loader:

- **Compatible with BCMI framework**: Extends `BaseBCMILoader`
- **BIDS format support**: Uses mne_bids for loading
- **Session-based organization**: Each of 12 sessions represents one song
- **Full song trials**: Each trial contains the complete EEG recording for one song
- **No audio files**: The dataset contains only EEG data and metadata
- **Behavioral data**: Ratings available separately in stimuli directory